# Stage 6.1.2 — Quality-Corrected Hardened Baseline Checkpoint

This notebook validates the pre-execution Stage 6.1.2 corrective package and, when a completed manual
baseline bundle is supplied, regenerates the candidate comparison from an immutable Git
reference. It does not treat `main` as a reproducible reference.


In [ ]:
REPO_URL = "https://github.com/richietrap/sbom_to_audit.git"
REF = "STAGE612_REF_REQUIRED"
MANUAL_RESULTS_ZIP = ""  # Optional: upload a completed canonical CSV/YAML bundle ZIP.

if REF in {"main", "master", "STAGE612_REF_REQUIRED"}:
    raise ValueError("Set REF to an exact Stage 6.1.2 commit SHA or immutable tag.")


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

work = Path("/content/stage612-checkpoint")
if work.exists():
    shutil.rmtree(work)
subprocess.run(["git", "clone", REPO_URL, str(work)], check=True)
os.chdir(work)
subprocess.run(["git", "checkout", REF], check=True)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Exact commit:", commit)


In [ ]:
subprocess.run(["python", "-m", "pip", "install", "-e", ".[dev]"], check=True)
subprocess.run(["python", "-m", "pip", "check"], check=True)
subprocess.run(["python", "-m", "compileall", "-q", "src", "scripts", "tests"], check=True)


In [ ]:
import sbom_to_audit
assert sbom_to_audit.__version__ == "0.6.1.2", sbom_to_audit.__version__
print("PASS: package version 0.6.1.2")
subprocess.run(["python", "scripts/validate_repository.py", "--strict-sources"], check=True)
subprocess.run(["python", "scripts/freeze_stage6_1_protocol.py", "--verify"], check=True)
subprocess.run(["python", "scripts/validate_stage6_1_evaluation.py"], check=True)
subprocess.run(["python", "-m", "pytest", "-q"], check=True)
subprocess.run(["python", "scripts/release_check.py"], check=True)


In [ ]:
packet_root = Path("/content/stage612-baseline-packets")
if packet_root.exists():
    shutil.rmtree(packet_root)
subprocess.run([
    "python", "scripts/export_stage6_1_baseline_packets.py",
    "--destination", str(packet_root),
], check=True)
print("Blinded packets:", packet_root)


In [ ]:
if MANUAL_RESULTS_ZIP:
    bundle_root = Path("/content/stage612-manual-results")
    if bundle_root.exists():
        shutil.rmtree(bundle_root)
    shutil.unpack_archive(MANUAL_RESULTS_ZIP, bundle_root)
    candidates = list(bundle_root.rglob("declaration.yaml"))
    if len(candidates) != 1:
        raise ValueError("Manual result ZIP must contain exactly one canonical bundle.")
    bundle = candidates[0].parent
    imported = Path("/content/stage612-imported")
    comparison = Path("/content/stage612-comparison")
    assets = Path("/content/stage612-paper-assets")
    subprocess.run([
        "python", "scripts/validate_manual_baseline_worksheet.py", str(bundle),
        "--require-complete",
    ], check=True)
    subprocess.run([
        "python", "scripts/import_manual_baseline_results.py", str(bundle),
        "--destination", str(imported),
    ], check=True)
    normalized = imported / "normalized" / "stage6_1_manual_baseline_normalized.json"
    subprocess.run([
        "python", "scripts/run_stage6_1_comparison.py", str(normalized),
        "--destination", str(comparison),
    ], check=True)
    report = comparison / "comparison" / "stage6_1_comparison_report.json"
    subprocess.run([
        "python", "scripts/validate_stage6_1_evaluation.py",
        "--comparison-report", str(report),
    ], check=True)
    subprocess.run([
        "python", "scripts/build_stage6_1_paper_assets.py",
        str(comparison / "comparison"), "--destination", str(assets),
    ], check=True)
else:
    print("Stage 6.1.2 pre-execution checkpoint complete; no manual result bundle supplied.")


In [ ]:
import hashlib
import json
import platform
import zipfile
from datetime import datetime, timezone

checkpoint_root = Path("/content/stage612_checkpoint_evidence")
if checkpoint_root.exists():
    shutil.rmtree(checkpoint_root)
checkpoint_root.mkdir()
report = {
    "checkpoint_id": "STAGE6-1-2-CHECKPOINT-001",
    "git_commit": commit,
    "ref": REF,
    "python": platform.python_version(),
    "platform": platform.platform(),
    "generated_at": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
    "manual_results_supplied": bool(MANUAL_RESULTS_ZIP),
    "stage": "6.1.1",
    "package_version": "0.6.1.2",
}
(checkpoint_root / "checkpoint_report.json").write_text(
    json.dumps(report, indent=2) + "\n", encoding="utf-8"
)
for path in [
    Path("evaluation/freeze/stage6_1_protocol_freeze.json"),
    Path("evaluation/baseline_protocol_v0.2.yaml"),
]:
    shutil.copy2(path, checkpoint_root / path.name)
if MANUAL_RESULTS_ZIP:
    shutil.copytree(Path("/content/stage612-imported"), checkpoint_root / "imported")
    shutil.copytree(Path("/content/stage612-comparison"), checkpoint_root / "comparison")
    shutil.copytree(Path("/content/stage612-paper-assets"), checkpoint_root / "paper_assets")
zip_path = Path("/content/stage612_colab_checkpoint_evidence.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(checkpoint_root.rglob("*")):
        if path.is_file():
            archive.write(path, path.relative_to(checkpoint_root))
print("Checkpoint ZIP:", zip_path)
print("SHA-256:", hashlib.sha256(zip_path.read_bytes()).hexdigest())
